# Trabalho de Processamento Digital de Sinais Biomédicos

Este notebook adapta o código disponível no repositório para execução no Google Colab. O arquivo de entrada contém o ECG de três canais amostrado a 1000 amostras/s. Atualmente, somente o item (a) está implementado.

In [ ]:
# Instale as dependências necessárias no ambiente do Colab.
%pip -q install numpy pandas matplotlib

In [ ]:
from pathlib import Path
from google.colab import files
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FREQUENCIA_ORIGINAL = 1000
FREQUENCIA_NOVA = 4000
ARQUIVO_ENTRADA = "ECG_XYZ.txt(fs=1000Hz)"

# Selecione ECG_XYZ.txt(fs=1000Hz) no seu computador.
arquivos_enviados = files.upload()
if ARQUIVO_ENTRADA not in arquivos_enviados:
    raise FileNotFoundError(
        f"Envie o arquivo com o nome exato {ARQUIVO_ENTRADA!r}."
    )
dados = pd.read_csv(io.BytesIO(arquivos_enviados[ARQUIVO_ENTRADA]), sep="\t")
dados.head()

## Item (a) — Interpolação para 4000 amostras/s

O enunciado pede a interpolação do sinal original e a comparação gráfica entre 3:00 e 3:10, marcando as amostras. A implementação existente usa interpolação linear (`numpy.interp`) em cada canal.

### Funções do item

A função abaixo recebe o DataFrame carregado e a frequência alvo. Retorna um DataFrame com `SamplingInterval` e os canais interpolados.

In [ ]:
def interpolar_dados(dados_origem, frequencia):
    """Interpola os canais de ECG para uma frequência de amostragem desejada.

    Args:
        dados_origem: DataFrame com a coluna ``SamplingInterval`` e os canais.
        frequencia: Frequência alvo, em amostras por segundo.

    Returns:
        DataFrame com o tempo e os canais interpolados linearmente.
    """
    tempo_original = dados_origem["SamplingInterval"].to_numpy()
    passo_novo = 1 / frequencia
    tempo_novo = np.arange(
        tempo_original[0], tempo_original[-1] + passo_novo / 2, passo_novo
    )
    resultado = {"SamplingInterval": tempo_novo}
    for coluna in dados_origem.columns[1:]:
        resultado[coluna] = np.interp(
            tempo_novo, tempo_original, dados_origem[coluna].to_numpy()
        )
    return pd.DataFrame(resultado)


def plotar_sinal_ecg(
    tempo_inicio, tempo_fim, frequencia, dados_originais, dados_interpolados
):
    """Exibe os sinais original e interpolado no intervalo solicitado.

    Args:
        tempo_inicio: Início do intervalo, em segundos.
        tempo_fim: Fim do intervalo, em segundos.
        frequencia: Frequência do sinal interpolado, em amostras por segundo.
        dados_originais: DataFrame com o ECG original e a coluna de tempo.
        dados_interpolados: DataFrame com o ECG interpolado e a coluna de tempo.

    Returns:
        None. Exibe a figura diretamente no notebook.
    """
    tempo_original = dados_originais["SamplingInterval"].to_numpy()
    tempo_interpolado = dados_interpolados["SamplingInterval"].to_numpy()
    mascara_original = (tempo_original >= tempo_inicio) & (tempo_original <= tempo_fim)
    mascara_interpolado = (tempo_interpolado >= tempo_inicio) & (tempo_interpolado <= tempo_fim)
    canais = [coluna for coluna in dados_interpolados.columns if coluna != "SamplingInterval"]
    cores = plt.get_cmap("tab10").colors
    fig, eixos = plt.subplots(
        nrows=len(canais), ncols=1, figsize=(12, 9), sharex=True,
        constrained_layout=True
    )
    if len(canais) == 1:
        eixos = [eixos]
    for indice, canal in enumerate(canais):
        eixo = eixos[indice]
        eixo.plot(
            tempo_original[mascara_original],
            dados_originais.loc[mascara_original, canal],
            color=cores[indice * 2], linestyle="None", marker="o", markersize=2,
            label="Original (1000 amostras/s)"
        )
        eixo.plot(
            tempo_interpolado[mascara_interpolado],
            dados_interpolados.loc[mascara_interpolado, canal],
            color=cores[indice * 2 + 1], linewidth=0.7,
            label=f"Interpolado ({frequencia} amostras/s)"
        )
        eixo.set_title(canal, loc="left", fontsize=10)
        eixo.set_ylabel("Amplitude")
        eixo.grid(True, alpha=0.3)
        eixo.legend(loc="upper right", fontsize=8)
    ticks = np.arange(np.ceil(tempo_inicio), np.floor(tempo_fim) + 1, 1)
    eixos[-1].set_xticks(ticks)
    eixos[-1].set_xticklabels([f"{int(t // 60)}:{int(t % 60):02d}" for t in ticks])
    eixos[-1].set_xlim(tempo_inicio, tempo_fim)
    eixos[-1].set_xlabel("Tempo (min:s)")
    fig.suptitle(
        f"ECG original e interpolado · {int(tempo_inicio // 60)}:"
        f"{int(tempo_inicio % 60):02d}–{int(tempo_fim // 60)}:"
        f"{int(tempo_fim % 60):02d} ({frequencia} amostras/s)"
    )
    plt.show()
    plt.close(fig)


### Execução e resultados

O trecho também mantém um CSV do sinal interpolado, como no fluxo do projeto. O gráfico é exibido diretamente no notebook e não é salvo como imagem.

In [ ]:
dados_interpolados = interpolar_dados(dados, FREQUENCIA_NOVA)

arquivo_saida = Path("ECG_XYZ_4000Hz.csv")
dados_interpolados.to_csv(arquivo_saida, index=False)
print(f"Sinal interpolado salvo em: {arquivo_saida}")

plotar_sinal_ecg(
    tempo_inicio=180, tempo_fim=190, frequencia=FREQUENCIA_NOVA,
    dados_originais=dados, dados_interpolados=dados_interpolados,
)

## Itens (b) a (g) — Ainda não implementados

O PDF também solicita detecção e promediação de complexos PQRST, intervalos RR, QT/QTc, estatísticas dos intervalos RR, seleção por faixa de dois desvios padrão e comparação espectral. Esses itens não possuem implementação no projeto atual e, portanto, não há células de cálculo correspondentes neste notebook.